In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
import tensorflow as tf

2024-06-16 16:45:35.488423: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-16 16:45:35.555244: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-16 16:45:35.555311: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-16 16:45:35.557024: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-06-16 16:45:35.566899: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-16 16:45:35.567987: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [6]:
with open("DATABASE/final_unpreprocessed/cleaned_combined_Elytis.txt") as f:
    data = f.read() 

data[:100] 

'T: ΤΑ  ΕΛΕΓΕΙΑ  ΤΗΣ ΟΞΩΠΕΤΡΑΣ   (1991)\nT: ΑΚΙΝΔΥΝΟΥ, ΕΛΠΙΔΟΦΟΡΟΥ , ΑΝΕΜΠΟΔΙΣΤΟΥ\nΤώρα, στη βάρκα όπου'

In [7]:
chunk_size =26
chunk_overlap = 4

In [8]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)
c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

In [9]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character",
standardize="lower")
text_vec_layer.adapt([data])
encoded = text_vec_layer([data])[0] 

2024-06-16 16:47:36.494531: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [10]:
dataset_size = len(encoded)
dataset_size 

531381

In [11]:
encoded -= 2  # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2  # number of distinct chars = 39
dataset_size = len(encoded) 

155

In [11]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    """It takes a sequence as input (i.e., the encoded text), and creates a dataset
    containing all the windows of the desired length.

    It increases the length by one, since we need the next character for the
    target.

    Then, it shuffles the windows (optionally), batches them, splits them
    into input/output pairs, and activates prefetching.
    """
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(buffer_size=100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [12]:

length = 100
tf.random.set_seed(42)

# Total length of the dataset
total_length = 531381

# Adjust the splits accordingly
train_end = int(0.8 * total_length)  # 80% for training
valid_end = int(0.9 * total_length)  # 10% for validation, 10% for testing

train_set = to_dataset(encoded[:train_end], length=length, shuffle=True, seed=42)
valid_set = to_dataset(encoded[train_end:valid_end], length=length)
test_set = to_dataset(encoded[valid_end:], length=length)

In [13]:
# Save datasets to disk
tf.data.experimental.save(train_set, "train_set")
tf.data.experimental.save(valid_set, "valid_set")
tf.data.experimental.save(test_set, "test_set")

print("Datasets saved successfully.")

Instructions for updating:
Use `tf.data.Dataset.save(...)` instead.


2024-06-16 16:10:57.047094: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] ShuffleDatasetV3:19: Filling up shuffle buffer (this may take a while): 42334 of 100000
2024-06-16 16:11:12.809796: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] Shuffle buffer filled.


Datasets saved successfully.
